In [2]:
import casadi as ca
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ============================================================
# 1. Параметры модели Лотки–Вольтерры и метода
# ============================================================
a = 1.0
b = 1.0
c = 1.0
d = 1.0

t0 = 0.0
tf = 20.0
N = 10                     # число конечных элементов (можно меньше благодаря высокому порядку)
h = (tf - t0) / N

# Коллокационные точки Radau IIA для K = 3 (порядок 5)
# τ₁, τ₂, τ₃=1
tau = np.array([0.1550510257216822,
                0.6449489742783178,
                1.0])
K = len(tau)               # K = 3

# ============================================================
# 2. Вычисление матрицы D и весов L0
# ============================================================
def lagrange_basis(tau, x):
    """Вычисляет лагранжевы базисные полиномы L_j(x) для точек tau."""
    L = np.zeros(len(tau))
    for j in range(len(tau)):
        prod = 1.0
        for m in range(len(tau)):
            if m != j:
                prod *= (x - tau[m]) / (tau[j] - tau[m])
        L[j] = prod
    return L

def differentiation_matrix(tau):
    """Матрица D_{kj} = dL_j/dτ (τ_k)."""
    K = len(tau)
    D = np.zeros((K, K))
    for k in range(K):
        for j in range(K):
            # производная L_j в точке tau_k
            sum_term = 0.0
            for m in range(K):
                if m != j:
                    prod = 1.0 / (tau[j] - tau[m])
                    for r in range(K):
                        if r != j and r != m:
                            prod *= (tau[k] - tau[r]) / (tau[j] - tau[r])
                    sum_term += prod
            D[k, j] = sum_term
    return D

D = differentiation_matrix(tau)
L0 = lagrange_basis(tau, 0.0)   # значения L_j(0)

print("Точки коллокации:\n", tau)
print("Матрица D:\n", D)
print("L0:\n", L0)

# ============================================================
# 3. Символьное построение NLP (коллокационные уравнения)
# ============================================================
# Тензор состояний X[элемент, точка, переменная] (x и y)
X = ca.SX.sym('X', N, K, 2)

p_a = ca.SX.sym('a')
p_b = ca.SX.sym('b')
p_c = ca.SX.sym('c')
p_d = ca.SX.sym('d')

x0_init = 1.2
y0_init = 1.2

# Правая часть ОДУ
def f_ode(x, y, a, b, c, d):
    dx = a*x - b*x*y
    dy = -c*y + d*x*y
    return ca.vertcat(dx, dy)

g = []
lbg = []
ubg = []

# --- Коллокационные уравнения ---
for i in range(N):
    for k in range(K):
        # производная по локальному времени τ
        dx_dtau = 0
        for j in range(K):
            dx_dtau += D[k, j] * X[i, j, :]
        xk = X[i, k, 0]
        yk = X[i, k, 1]
        f_val = f_ode(xk, yk, p_a, p_b, p_c, p_d)
        g.append(dx_dtau[0] - h * f_val[0])
        lbg.append(0); ubg.append(0)
        g.append(dx_dtau[1] - h * f_val[1])
        lbg.append(0); ubg.append(0)

# --- Начальное условие ---
x_first_left = ca.dot(L0, X[0, :, 0])
y_first_left = ca.dot(L0, X[0, :, 1])
g.append(x_first_left - x0_init)
lbg.append(0); ubg.append(0)
g.append(y_first_left - y0_init)
lbg.append(0); ubg.append(0)

# --- Условия непрерывности ---
for i in range(N-1):
    # правый конец i-го элемента – последняя коллокационная точка (τ=1)
    x_right = X[i, K-1, 0]
    y_right = X[i, K-1, 1]
    # левый конец (i+1)-го элемента – интерполяция в τ=0
    x_next_left = ca.dot(L0, X[i+1, :, 0])
    y_next_left = ca.dot(L0, X[i+1, :, 1])
    g.append(x_next_left - x_right)
    lbg.append(0); ubg.append(0)
    g.append(y_next_left - y_right)
    lbg.append(0); ubg.append(0)

# ============================================================
# 4. Решение NLP
# ============================================================
nvars = N * K * 2
X_vars = X.reshape((-1, 1))
p = ca.vertcat(p_a, p_b, p_c, p_d)
opt_vars = ca.vertcat(X_vars, p)

# параметры фиксированы
lbx = np.concatenate([-np.inf * np.ones(nvars), np.array([a, b, c, d])])
ubx = np.concatenate([ np.inf * np.ones(nvars), np.array([a, b, c, d])])

nlp = {'x': opt_vars, 'f': 0, 'g': ca.vertcat(*g)}
opts = {'ipopt.print_level': 0, 'print_time': 0}
solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

# начальное приближение состояний (грубая линейная интерполяция)
t_grid = np.linspace(t0, tf, N*K+1)[:-1]   # моменты всех коллокационных точек
x_guess = np.interp(t_grid, [t0, tf], [x0_init, 0.5])
y_guess = np.interp(t_grid, [t0, tf], [y0_init, 1.5])
init_guess = np.concatenate([x_guess, y_guess, a, b, c, d])

sol = solver(x0=init_guess, lbx=lbx, ubx=ubx, lbg=lbg, ubg=ubg)
opt = sol['x'].full().flatten()
X_opt = opt[:nvars].reshape(N, K, 2)
print("Решение NLP найдено.")

# ============================================================
# 5. Восстановление траектории и сравнение с эталоном
# ============================================================
# Эталонное решение scipy (RK45 с высокой точностью)
def lotka_volterra(t, z):
    x, y = z
    return [a*x - b*x*y, -c*y + d*x*y]
sol_ref = solve_ivp(lotka_volterra, [t0, tf], [x0_init, y0_init],
                    method='RK45', rtol=1e-12, atol=1e-14, dense_output=True)

# Мелкая сетка для визуализации
t_plot = np.linspace(t0, tf, 500)
x_plot = np.zeros_like(t_plot)
y_plot = np.zeros_like(t_plot)
x_ref = sol_ref.sol(t_plot)[0]
y_ref = sol_ref.sol(t_plot)[1]

for idx, t in enumerate(t_plot):
    i = min(int(np.floor((t - t0) / h)), N-1)
    tau_loc = (t - (t0 + i*h)) / h
    L_val = lagrange_basis(tau, tau_loc)   # используем ту же функцию
    x_plot[idx] = np.dot(L_val, X_opt[i, :, 0])
    y_plot[idx] = np.dot(L_val, X_opt[i, :, 1])

# Ошибка
err_x = np.max(np.abs(x_plot - x_ref))
err_y = np.max(np.abs(y_plot - y_ref))
print(f"Максимальная ошибка по x: {err_x:.2e}")
print(f"Максимальная ошибка по y: {err_y:.2e}")

# График
plt.figure(figsize=(10, 5))
plt.subplot(1,2,1)
plt.plot(t_plot, x_plot, 'b-', label='OCFE (K=3)')
plt.plot(t_plot, x_ref, 'r--', label='RK45 эталон')
plt.xlabel('Время')
plt.ylabel('x (жертвы)')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(t_plot, y_plot, 'b-', label='OCFE (K=3)')
plt.plot(t_plot, y_ref, 'r--', label='RK45 эталон')
plt.xlabel('Время')
plt.ylabel('y (хищники)')
plt.legend()
plt.grid(True)

plt.suptitle('Лотка–Вольтерра: Radau IIA, K=3 (порядок 5)')
plt.tight_layout()
plt.show()

Точки коллокации:
 [0.15505103 0.64494897 1.        ]
Матрица D:
 [[-3.22474487  4.85773803 -1.63299316]
 [-0.85773803 -0.77525513  1.63299316]
 [ 0.85773803 -4.85773803  4.        ]]
L0:
 [ 1.5580782  -0.89141154  0.33333333]


TypeError: list indices must be integers or slices, not tuple